
## Scenario: ICP Consecutive Daily Billing

**Description:** Active claims where an ICP (Informal Care Provider) has billed on 30 or more consecutive days within the last 95 days.

In [0]:
ENGINE_CATALOG = dbutils.widgets.get("ENGINE_CATALOG")
ENGINE_SCHEMA = dbutils.widgets.get("ENGINE_SCHEMA")

GRAPH_CATALOG = dbutils.widgets.get("GRAPH_CATALOG")
GRAPH_SCHEMA = dbutils.widgets.get("GRAPH_SCHEMA")

In [0]:
%sql

DECLARE execDatetime TIMESTAMP = GETDATE();

In [0]:
df = spark.sql(f"""
    SELECT
        NOVEL_SCENARIO_ID
    FROM
        {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_NOVEL_SCENARIO
    WHERE
        NOTEBOOK_NAME = 'Scenario_ICPConsecutiveDailyBilling'
""")

novelScenarioId = df.collect()[0][0]
print(f"Novel scenario ID: {novelScenarioId}")

In [0]:
# Number of calendar days to look back from today when searching for charges
windowDays = 95
# Minimum number of consecutive billed days required to flag a (claim, provider) pair
minConsecutiveDays = 30

In [0]:
spark.sql(f"""
SELECT
    CLAIM_ID,
    PROVIDER_ID,
    MAX(PROVIDER_CODE_AS)  AS PROVIDER_CODE_AS,
    MAX(INVOICE_VALID_IND) AS INVOICE_VALID_IND,
    CHARGE_DATE,
    SUM(CHARGE_AMT)        AS DAILY_CHARGE_AMT
FROM 
    {GRAPH_CATALOG}.gold_buffer_driver.T_DRVR_CHARGE_PAYMENT
WHERE 
    INVOICE_VALID_IND = 1
    AND PROVIDER_CODE_AS = 'ICP / Informal'
    AND CHARGE_AMT > 1
    AND CHARGE_DATE BETWEEN DATE_SUB(GETDATE(), {windowDays}) AND GETDATE()
GROUP BY 
    CLAIM_ID, PROVIDER_ID, CHARGE_DATE
""").createOrReplaceTempView("charges_daily")

In [0]:
spark.sql("""
SELECT
    CLAIM_ID,
    PROVIDER_ID,
    PROVIDER_CODE_AS,
    CHARGE_DATE,
    DAILY_CHARGE_AMT,
    -- On consecutive days, CHARGE_DATE increments by 1 and so does ROW_NUMBER,
    -- so their difference stays constant — forming the 'island' key.
    -- A gap in billing shifts the date by more than 1 while ROW_NUMBER
    -- only increments by 1, producing a new constant and starting a new island.
    DATE_SUB(
        CHARGE_DATE,
        ROW_NUMBER() OVER (PARTITION BY CLAIM_ID, PROVIDER_ID ORDER BY CHARGE_DATE)
    ) AS RUN_KEY
FROM 
    charges_daily
""").createOrReplaceTempView("charges_grouped")

In [0]:
# Collapse each consecutive run into one summary row.
# Only keep runs that meet the minimum consecutive-days threshold.
spark.sql(f"""
SELECT
    CLAIM_ID,
    PROVIDER_ID,
    MAX(PROVIDER_CODE_AS) AS PROVIDER_CODE_AS,
    MIN(CHARGE_DATE)      AS RUN_START,
    MAX(CHARGE_DATE)      AS RUN_END,
    COUNT(*)              AS CONSECUTIVE_DAYS,
    SUM(DAILY_CHARGE_AMT) AS TOTAL_CHARGE_AMT
FROM 
    charges_grouped
GROUP BY 
    CLAIM_ID, PROVIDER_ID, RUN_KEY
HAVING 
    COUNT(*) >= {minConsecutiveDays}
""").createOrReplaceTempView("consecutive_runs")

In [0]:
# For each claim, keep only the single longest consecutive run.
# Ties are broken by most-recent end date, then highest total charge amount.
spark.sql("""
SELECT
    r.CLAIM_ID,
    r.PROVIDER_ID,
    r.PROVIDER_CODE_AS,
    r.RUN_START,
    r.RUN_END,
    r.CONSECUTIVE_DAYS,
    ROUND(r.TOTAL_CHARGE_AMT, 2) AS TOTAL_CHARGE_AMT_IN_RUN
FROM 
    consecutive_runs r
JOIN (
    SELECT 
        CLAIM_ID, 
        MAX(CONSECUTIVE_DAYS) AS MAX_CONSECUTIVE_DAYS
    FROM 
        consecutive_runs
    GROUP BY 
        CLAIM_ID
) b
    ON r.CLAIM_ID = b.CLAIM_ID
    AND r.CONSECUTIVE_DAYS = b.MAX_CONSECUTIVE_DAYS
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.CLAIM_ID
    ORDER BY r.RUN_END DESC, r.TOTAL_CHARGE_AMT DESC
) = 1
""").createOrReplaceTempView("best_run_per_claim")

In [0]:
spark.sql(f"""
SELECT
    CLAIM_ID,
    CLAIM_NUMBER,
    CLAIM_STATUS_CODE,
    CLAIM_OPEN_DATE
FROM 
    {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_CLAIM
WHERE 
    CLAIM_STATUS_CODE IN ('Active', 'Benefit Period', 'Qualification Period', 'ASWP')
""").createOrReplaceTempView("active_claims")

In [0]:
spark.sql("""
SELECT
    c.CLAIM_ID,
    c.CLAIM_NUMBER,
    c.CLAIM_STATUS_CODE,
    c.CLAIM_OPEN_DATE,
    b.PROVIDER_ID,
    b.PROVIDER_CODE_AS,
    b.CONSECUTIVE_DAYS      AS MAX_CONSECUTIVE_DAYS,
    b.RUN_START,
    b.RUN_END,
    b.TOTAL_CHARGE_AMT_IN_RUN,
    execDatetime            AS FEATURE_DATETIME
FROM 
    best_run_per_claim b
JOIN 
    active_claims c
    ON b.CLAIM_ID = c.CLAIM_ID
ORDER BY 
    MAX_CONSECUTIVE_DAYS DESC, c.CLAIM_NUMBER
""").createOrReplaceTempView("t_flagged_claims")

In [0]:
spark.sql(f"""
    INSERT INTO 
        {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_SCENARIO_ICP_CONSECUTIVE_DAILY_BILLING_DETAIL
    SELECT * FROM 
        t_flagged_claims
""")